이 노트북은 <[케라스 창시자에게 배우는 딥러닝 (개정 3판)](https://tensorflow.blog/kerasdl3/)>(길벗, 2026)의 예제 코드를 담고 있습니다.

### JAX 소개

**준비:** 원본 노트북의 텐서플로 절에서 만든 데이터를 이 노트북에서도 쓰므로 여기서 다시 만듭니다.

In [1]:
import numpy as np

num_samples_per_class = 1000
negative_samples = np.random.multivariate_normal(
    mean=[0, 3], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)
positive_samples = np.random.multivariate_normal(
    mean=[3, 0], cov=[[1, 0.5], [0.5, 1]], size=num_samples_per_class
)

In [2]:
inputs = np.vstack((negative_samples, positive_samples)).astype(np.float32)

In [3]:
targets = np.vstack(
    (
        np.zeros((num_samples_per_class, 1), dtype="float32"),
        np.ones((num_samples_per_class, 1), dtype="float32"),
    )
)

In [4]:
x = np.ones(shape=(2, 2))

#### JAX 첫걸음

#### JAX의 텐서

In [5]:
from jax import numpy as jnp
jnp.ones(shape=(2, 1))

Array([[1.],
       [1.]], dtype=float32)

In [6]:
jnp.zeros(shape=(2, 1))

Array([[0.],
       [0.]], dtype=float32)

In [7]:
jnp.array([1, 2, 3], dtype="float32")

Array([1., 2., 3.], dtype=float32)

#### JAX의 난수 생성

In [8]:
np.random.normal(size=(3,))

array([-1.21330946,  0.72947474,  0.8436742 ])

In [9]:
np.random.normal(size=(3,))

array([-0.22059933, -1.02715021, -0.58273216])

In [10]:
def apply_noise(x, seed):
    np.random.seed(seed)
    x = x * np.random.normal((3,))
    return x

seed = 1337
y = apply_noise(x, seed)
seed += 1
z = apply_noise(x, seed)

In [11]:
import jax

seed_key = jax.random.key(1337)

In [12]:
seed_key = jax.random.key(0)
jax.random.normal(seed_key, shape=(3,))

Array([ 1.6226422 ,  2.0252647 , -0.43359435], dtype=float32)

In [13]:
seed_key = jax.random.key(123)
jax.random.normal(seed_key, shape=(3,))

Array([1.6359469 , 0.8408094 , 0.02212393], dtype=float32)

In [14]:
jax.random.normal(seed_key, shape=(3,))

Array([1.6359469 , 0.8408094 , 0.02212393], dtype=float32)

In [15]:
seed_key = jax.random.key(123)
jax.random.normal(seed_key, shape=(3,))

Array([1.6359469 , 0.8408094 , 0.02212393], dtype=float32)

In [16]:
new_seed_key = jax.random.split(seed_key, num=1)[0]
jax.random.normal(new_seed_key, shape=(3,))

Array([-0.4909335, -0.9478693, -1.775197 ], dtype=float32)

##### 텐서 할당

In [17]:
x = jnp.array([1, 2, 3], dtype="float32")
new_x = x.at[0].set(10)

##### 텐서 연산: JAX로 수학 계산하기

In [18]:
a = jnp.ones((2, 2))
b = jnp.square(a)
c = jnp.sqrt(a)
d = b + c
e = jnp.matmul(a, b)
e *= d

In [19]:
def dense(inputs, W, b):
    return jax.nn.relu(jnp.matmul(inputs, W) + b)

##### JAX로 그레이디언트 계산하기

In [20]:
def compute_loss(input_var):
    return jnp.square(input_var)

In [21]:
grad_fn = jax.grad(compute_loss)

In [22]:
input_var = jnp.array(3.0)
grad_of_loss_wrt_input_var = grad_fn(input_var)

##### JAX 그레이디언트 계산의 모범 사례

###### 손실 값 반환

In [23]:
grad_fn = jax.value_and_grad(compute_loss)
output, grad_of_loss_wrt_input_var = grad_fn(input_var)

###### 복잡한 함수에 대한 그레이디언트 구하기

###### 보조 출력 반환

##### @jax.jit로 JAX 함수 속도 높이기

In [24]:
@jax.jit
def dense(inputs, W, b):
    return jax.nn.relu(jnp.matmul(inputs, W) + b)

#### 엔드-투-엔드 예제: 순수 JAX로 구현하는 선형 분류기

In [25]:
def model(inputs, W, b):
    return jnp.matmul(inputs, W) + b

def mean_squared_error(targets, predictions):
    per_sample_losses = jnp.square(targets - predictions)
    return jnp.mean(per_sample_losses)

In [26]:
def compute_loss(state, inputs, targets):
    W, b = state
    predictions = model(inputs, W, b)
    loss = mean_squared_error(targets, predictions)
    return loss

In [27]:
grad_fn = jax.value_and_grad(compute_loss)

In [28]:
learning_rate = 0.1

@jax.jit
def training_step(inputs, targets, W, b):
    loss, grads = grad_fn((W, b), inputs, targets)
    grad_wrt_W, grad_wrt_b = grads
    W = W - grad_wrt_W * learning_rate
    b = b - grad_wrt_b * learning_rate
    return loss, W, b

In [29]:
input_dim = 2
output_dim = 1

W = jax.numpy.array(np.random.uniform(size=(input_dim, output_dim)))
b = jax.numpy.array(np.zeros(shape=(output_dim,)))
state = (W, b)
for step in range(40):
    loss, W, b = training_step(inputs, targets, W, b)
    print(f"{step}번째 스텝의 손실: {loss:.4f}")

0번째 스텝의 손실: 3.3716
1번째 스텝의 손실: 0.1885
2번째 스텝의 손실: 0.1112
3번째 스텝의 손실: 0.0962
4번째 스텝의 손실: 0.0893
5번째 스텝의 손실: 0.0839
6번째 스텝의 손실: 0.0791
7번째 스텝의 손실: 0.0747
8번째 스텝의 손실: 0.0706
9번째 스텝의 손실: 0.0668
10번째 스텝의 손실: 0.0634
11번째 스텝의 손실: 0.0602
12번째 스텝의 손실: 0.0573
13번째 스텝의 손실: 0.0546
14번째 스텝의 손실: 0.0522
15번째 스텝의 손실: 0.0499
16번째 스텝의 손실: 0.0478
17번째 스텝의 손실: 0.0459
18번째 스텝의 손실: 0.0442
19번째 스텝의 손실: 0.0426
20번째 스텝의 손실: 0.0411
21번째 스텝의 손실: 0.0397
22번째 스텝의 손실: 0.0385
23번째 스텝의 손실: 0.0373
24번째 스텝의 손실: 0.0363
25번째 스텝의 손실: 0.0353
26번째 스텝의 손실: 0.0344
27번째 스텝의 손실: 0.0336
28번째 스텝의 손실: 0.0328
29번째 스텝의 손실: 0.0321
30번째 스텝의 손실: 0.0315
31번째 스텝의 손실: 0.0309
32번째 스텝의 손실: 0.0304
33번째 스텝의 손실: 0.0299
34번째 스텝의 손실: 0.0294
35번째 스텝의 손실: 0.0290
36번째 스텝의 손실: 0.0286
37번째 스텝의 손실: 0.0282
38번째 스텝의 손실: 0.0279
39번째 스텝의 손실: 0.0276


#### JAX 방식의 차이점